# Abstract Composable SIEM × Microsoft Sentinel — Total Comparison Notebook

Runnable, side-by-side comparison that pulls **live Abstract API** data and **live Sentinel** data (via Azure CLI) and
quantifies cost/TCO, pipeline reduction (raw→enriched/aggregated), detections & MITRE, incidents/MTTR/MTTD, and AI-SOC.

**Prereqs:** `pip install requests pandas matplotlib` and `az login` — that's it.

The **Vendor Account ID** and **Workspace ID** are saved as defaults in the config cell, and the **Abstract API key is
fetched at runtime from Azure Key Vault** (`kv-abs-507fb9e4`, secret `abstract-api-key`) via your `az login` — so the
key is never stored in this notebook or on disk, and it survives rotation (just update the Key Vault secret). Override any
value with env vars (`ABSTRACT_API_KEY`, `ABSTRACT_VENDOR_ACCOUNT_ID`, `LA_WORKSPACE_ID`, `ABSTRACT_KEY_VAULT`) if needed.

> $/% comparisons use the parameters in the config cell (Sentinel list pricing, East US, verified 2026-07). Capability
> claims are grounded in Abstract content + Sentinel product facts.


## 1 · Config & helpers


In [ ]:
import os, json, subprocess, requests, pandas as pd, matplotlib.pyplot as plt

# --- Saved connection (non-secret IDs as defaults; API key from Key Vault via az login) ---
ABS_BASE   = os.environ.get('ABSTRACT_BASE_URL', 'https://api.abstractsecurity.app')
ABS_VENDOR = os.environ.get('ABSTRACT_VENDOR_ACCOUNT_ID', '12jW5BDyQR')          # Abstract vendor account id
LA_WSID    = os.environ.get('LA_WORKSPACE_ID', 'a890d458-d4e9-4590-9a1d-4b410b5dbbd2')  # Log Analytics customerId
KEY_VAULT  = os.environ.get('ABSTRACT_KEY_VAULT', 'kv-abs-507fb9e4')

def _kv_secret(name):
    return subprocess.check_output(['az','keyvault','secret','show','--vault-name',KEY_VAULT,'--name',name,'--query','value','-o','tsv']).decode().strip()

# API key: env var override, else fetched from Key Vault at runtime (never written to disk; rotation-proof)
ABS_KEY = os.environ.get('ABSTRACT_API_KEY') or _kv_secret('abstract-api-key')

# --- Sentinel list pricing (East US, verified 2026-07; adjust to your rate/region) ---
PRICE = {'analytics': 4.30, 'basic': 0.50, 'lake': 0.05, 'lake_storage_gb_mo': 0.026}
REDUCTION_PCT   = 0.70    # Abstract reports 70-80%
ANALYST_RATE    = 75.0    # $/hr
MIN_PER_ALERT   = 20      # minutes to triage
FP_PCT          = 0.40    # false positives among alerts
SOURCE_GB_DAY   = 500.0   # pre-Abstract daily source volume

def abstract(path, method='GET', body=None):
    h = {'Authorization': f'Bearer {ABS_KEY}', 'X-AS-Vendor-Account-ID': ABS_VENDOR, 'accept': 'application/json'}
    if method == 'GET':
        r = requests.get(ABS_BASE+path, headers=h, timeout=45)
    else:
        h['Content-Type']='application/json'; r = requests.post(ABS_BASE+path, headers=h, json=body, timeout=45)
    r.raise_for_status(); return r.json() if r.text else {}

def kql(query):
    out = subprocess.check_output(['az','monitor','log-analytics','query','-w',LA_WSID,'--analytics-query',query,'-o','json'])
    return pd.DataFrame(json.loads(out))

print('Abstract:', abstract('/v2/auth/').get('email'), '| Sentinel workspace:', LA_WSID)


## 2 · Cost & TCO across Sentinel tiers


In [ ]:
cur = SOURCE_GB_DAY*365*PRICE['analytics']
with_abs = SOURCE_GB_DAY*(1-REDUCTION_PCT)*365*PRICE['analytics'] + SOURCE_GB_DAY*REDUCTION_PCT*365*PRICE['lake']
tco = pd.DataFrame({'Scenario':['Sentinel-only (all Analytics)','With Abstract (hot+lake)'],
                    'Annual $':[round(cur), round(with_abs)]})
print(tco.to_string(index=False))
print(f'Annual savings: ${round(cur-with_abs):,}   |   3-year: ${round((cur-with_abs)*3):,}')
tco.plot.bar(x='Scenario', y='Annual $', legend=False, title='Annual Sentinel ingestion cost', rot=0); plt.tight_layout(); plt.show()

# Your actual billable ingestion by table
usage = kql('Usage | where IsBillable==true | summarize GB=round(sum(Quantity)/1024.0,3) by DataType | top 12 by GB desc')
display(usage)


## 3 · Pipeline reduction — raw → enriched & aggregated (measured)


In [ ]:
red = kql('AbstractEventLogs_CL | extend agg=todouble(coalesce(toreal(AbstractEvent.aggregation_count),1.0)), b=todouble(estimate_data_size(AbstractEvent)) | summarize Ingested=count(), Raw=sum(agg), Bytes=sum(b)')
if not red.empty:
    ing=float(red.Ingested[0]); raw=float(red.Raw[0]); avgb=float(red.Bytes[0])/max(ing,1)
    ratio=round(raw/max(ing,1),1); gb_avoided=round((raw-ing)*avgb/1024**3,3)
    print(f'Ingested={int(ing):,}  Raw represented={int(raw):,}  Aggregation ratio={ratio}:1  GB avoided={gb_avoided}  $ saved@Analytics=${round(gb_avoided*PRICE["analytics"],2)}')
    pd.Series({'Raw represented':raw,'Ingested':ing}).plot.bar(title=f'Raw vs ingested ({ratio}:1)', rot=0); plt.tight_layout(); plt.show()

# Enrichment coverage (OCSF)
enr = kql("AbstractEventLogs_CL | extend e=AbstractEvent | summarize Total=count(), R=countif(toreal(e.risk_score)>0), T=countif(array_length(todynamic(e.tags))>0)")
display(enr)
print('ACS schema fields (OCSF breadth):', len(abstract('/v1/acs/fields')))


## 4 · Detections, MITRE & AI-SOC (ASTRO)


In [ ]:
ins = abstract('/v1/insights/?page_number=1&page_size=200').get('insights', [])
idf = pd.DataFrame(ins)
print('Live Abstract insights:', len(idf))
if len(idf):
    display(idf['severity'].value_counts())
    tech = pd.Series([m.get('id') for i in ins for m in (i.get('mitre_attack_techniques') or [])]).value_counts().head(10)
    tech.plot.bar(title='Live Abstract insights by MITRE technique', rot=45); plt.tight_layout(); plt.show()

alerts = kql('SecurityAlert | summarize n=count() by AlertName | top 10 by n desc')
display(alerts)


## 5 · Incidents, MTTR & MTTD


In [ ]:
inc = kql('SecurityIncident | summarize n=count() by Severity, Status')
display(inc)
mttr = kql('SecurityIncident | where isnotempty(ClosedTime) | extend TTR=ClosedTime-CreatedTime | summarize AvgHrs=round(avg(TTR)/1h,2), Median=round(percentile(TTR/1h,50),2), Closed=count()')
display(mttr)
mttd = kql("SecurityAlert | where AlertName startswith 'Abstract' | extend lag=TimeGenerated-todatetime(StartTime) | summarize Alerts=count(), AvgSiemLagMin=round(avg(lag)/1m,1)")
print('SIEM detect lag vs Abstract in-stream (~0.5s):'); display(mttd)


## 6 · Pipeline control — tune-at-source filters (live)


In [ ]:
tf = abstract('/v2/rule-tuning-filters/').get('items', [])
print('Active tune-at-source filters:', len(tf))
pd.DataFrame([{'Title':t.get('title'),'AppliedRules':t.get('applied_rule_count'),
               'Conditions':len((t.get('tuning_filter_combination') or {}).get('conditions') or [])} for t in tf])


## 7 · Scorecard — better together

| Dimension | Abstract (Composable SIEM) | Sentinel (SIEM) |
| --- | --- | --- |
| Cost | reduce/route before ingest (70–80%); tier to data lake | per-GB Analytics $4.30 |
| Reduction | aggregate/dedupe/filter in-stream (measured ratio above) | limited native dedup |
| Detection | in-stream + historical + federated, portable | scheduled, post-ingest |
| Enrichment | 100% in-stream (OCSF, 473 fields) | connectors/playbooks post-ingest |
| AI-SOC | ASTRO embedded (agentic) | Copilot add-on |
| Tune-at-source | yes (filters above) | tune post-ingest, still billed |

*Run against your own tenant + workspace to produce a customer-specific outbrief.*
